### Multiple primary sources test data

Adds `location_id_aliases`, a second primary configuration stored under the alias IDs, and
`configuration_pairs` to the local KinD warehouse, then checks that
`ev.download.evaluation_subset()` picks up the alias series through the API.

Run after `01`-`03`. Requires a teehr version with `ev.download.location_id_aliases()`.
Writes use `upsert`, so the notebook can be re-run.

In [ ]:
import tempfile
from getpass import getpass
from pathlib import Path

import pandas as pd
from pyspark.sql import functions as F

from teehr import LocalReadWriteEvaluation, RemoteReadWriteEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session

#### Start the spark session

In [ ]:
spark = create_spark_session()

#### Initialize a TEEHR Evaluation with read/write privileges

In [ ]:
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

#### Location ID aliases

An alias can't already be a location ID, so these use a synthetic `testalt-` prefix on gages
with full observation records.

In [ ]:
alias_sites = ["01347000", "03068800", "08313000"]

aliases = pd.DataFrame({
    "location_id": [f"usgs-{s}" for s in alias_sites],
    "location_id_alias": [f"testalt-{s}" for s in alias_sites],
})
ev.location_id_aliases.load_dataframe(aliases, write_mode="upsert")
ev.location_id_aliases.to_pandas()

#### Second primary source stored under the aliases

Copies May 2020 `usgs_observations` (the window covered by the local `nwm30_retrospective`
data) to the alias IDs as `test_alt_observations`.

In [ ]:
ev.configurations.load_dataframe(
    pd.DataFrame({
        "name": ["test_alt_observations"],
        "timeseries_type": ["primary"],
        "description": ["Synthetic second primary source for alias testing"],
    }),
    write_mode="upsert",
)

alt_ts = (
    ev.primary_timeseries.to_sdf()
    .filter(F.col("configuration_name") == "usgs_observations")
    .filter(F.col("location_id").isin([f"usgs-{s}" for s in alias_sites]))
    .filter(F.col("value_time").between("2020-05-01", "2020-06-01"))
    .withColumn("location_id", F.regexp_replace("location_id", "^usgs-", "testalt-"))
    .withColumn("configuration_name", F.lit("test_alt_observations"))
)
ev.primary_timeseries.load_dataframe(alt_ts, write_mode="upsert")

#### Configuration pairs

Pairs each secondary configuration in the joined tables with `usgs_observations`, which keeps
`test_alt_observations` out of the existing joins, then pairs `nwm30_retrospective` with
`test_alt_observations` too. The last pair changes what a regenerated `sim_joined_timeseries`
contains; drop it if the local metrics should stay as they are.

In [ ]:
secondaries = [
    "nwm30_retrospective",
    "nwm30_medium_range",
    "nwm30_short_range_alaska",
    "nwm30_short_range_hawaii",
    "nwm30_short_range_puertorico",
]
pairs = pd.DataFrame({
    "primary_configuration_name": ["usgs_observations"] * len(secondaries) + ["test_alt_observations"],
    "secondary_configuration_name": secondaries + ["nwm30_retrospective"],
})
ev.configuration_pairs.load_dataframe(pairs, write_mode="upsert")
ev.configuration_pairs.to_pandas()

#### Check the download

Downloads a subset into a temporary local evaluation through the in-cluster API. Paste an API key
when prompted (create one on the dashboards admin page).
Pass the `locations` ID; the alias series is found through `location_id_aliases`.

In [ ]:
dl_ev = LocalReadWriteEvaluation(
    dir_path=Path(tempfile.mkdtemp()) / "alias_download_test",
    create_dir=True,
    spark=spark,
)
dl_ev.download.configure(
    api_base_url="http://teehr-api:8000",
    api_key=getpass("TEEHR API key: "),
)

dl_ev.download.evaluation_subset(
    location_ids=["usgs-01347000"],
    primary_configuration_name="test_alt_observations",
    secondary_configuration_name="nwm30_retrospective",
    start_date="2020-05-01",
    end_date="2020-06-01",
)

In [ ]:
print(dl_ev.location_id_aliases.to_pandas())
print(dl_ev.configuration_pairs.to_pandas())
dl_ev.primary_timeseries.to_pandas().groupby(["location_id", "configuration_name"]).size()